In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("train.csv")
df = df.drop(columns = ["ID"])

In [ ]:
Mid_classes = [10,11,14]
Hard_classes = [0, 3, 9, 15, 16, 19]

In [ ]:
df_hardclasses = df[df["target"].isin(Mid_classes + Hard_classes)].copy()

In [ ]:
df_hardclasses["target"] = df_hardclasses["target"].apply(lambda x: 0 if x in Hard_classes else x)

In [ ]:
#df_hardclasses = df.query("target in @Hard_classes")

## 실험

In [ ]:
random_state = 45

In [ ]:
# 입력(X), 출력(y) 분리
X = df_hardclasses.drop(columns=["target"])  # 독립변수
y = df_hardclasses["target"]                 # 종속변수

le = LabelEncoder()
y_enc = le.fit_transform(y)

# Train/Test Split (8:2 비율)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.3, random_state=random_state, stratify=y  # stratify=y는 클래스 비율 유지
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
                              columns=X_train.columns, index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),
                              columns=X_test.columns, index=X_test.index)

y_test = le.inverse_transform(y_test)

In [ ]:
y_test = le.transform(y_test)

### 1차

- Scaling_only

In [ ]:
X_train_type1   = X_train_scaled
X_test_type1    = X_test_scaled

model_type1 = XGBClassifier(
    objective="multi:softprob",
    num_class=6,  # 클래스 개수
    max_depth=4,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=random_state
)

model_type1.fit(X_train_type1, y_train)


y_pred_enc_type1 = model_type1.predict(X_test_type1)
y_pred_type1 = le.inverse_transform(y_pred_enc_type1)


cm_type1 = confusion_matrix(y_test, y_pred_type1)
macro_f1 = f1_score(y_test, y_pred_type1, average="macro")
print("Macro F1 Score:", macro_f1)


# predict_proba: 각 샘플이 각 클래스일 확률
proba = model_type1.predict_proba(X_test)  # shape: (n_samples, n_classes)

class_labels = model_type1.classes_  # 모델이 학습한 클래스 레이블
proba_df_type1 = pd.DataFrame(proba, columns=[f"prob_{c}" for c in class_labels], index=X_test.index)
proba_df_type1["true_label"] = y_test

- Scaled_PCA

In [ ]:
# ============== PCA        ==============
# ============== 52차원      ==============
n_comps = 52

pca = PCA(n_components=n_comps)
X_train_type2 = pd.DataFrame(pca.fit_transform(X_train_scaled),
                           columns=[f"PC{i+1}" for i in range(n_comps)],
                           index=X_train.index)

X_test_type2  = pd.DataFrame(pca.transform(X_test_scaled),
                           columns=[f"PC{i+1}" for i in range(n_comps)],
                           index=X_test.index)


model_type2 = XGBClassifier(
    objective="multi:softprob",
    num_class=6,  # 클래스 개수
    max_depth=4,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=random_state
)

model_type2.fit(X_train_type2, y_train)

y_pred_enc_type2    = model_type2.predict(X_test_type2)
y_pred_type2 = le.inverse_transform(y_pred_enc_type2)


cm_type2        = confusion_matrix(y_test, y_pred_type2)
macro_f1        = f1_score(y_test, y_pred_type2, average="macro")
print("Macro F1 Score:", macro_f1)

# predict_proba: 각 샘플이 각 클래스일 확률
proba = model_type2.predict_proba(X_test_type2)  # shape: (n_samples, n_classes)

class_labels = model_type2.classes_  # 모델이 학습한 클래스 레이블
proba_df_type2 = pd.DataFrame(proba, columns=[f"prob_{c}" for c in class_labels], index=X_test.index)
proba_df_type2["true_label"] = y_test

- Scaled_LDA

In [ ]:
# ============== LDA           ==============
# ============== 최대5차원      ==============
n_lda_comps = 3
lda = LinearDiscriminantAnalysis(n_components=n_lda_comps)

X_train_type3 = pd.DataFrame(lda.fit_transform(X_train_scaled, y_train),
                           columns=[f"LDA{i+1}" for i in range(n_lda_comps)],
                           index=X_train.index)
X_test_type3  = pd.DataFrame(lda.transform(X_test_scaled),
                           columns=[f"LDA{i+1}" for i in range(n_lda_comps)],
                           index=X_test.index)


model_type3 = XGBClassifier(
    objective="multi:softprob",
    num_class=6,  # 클래스 개수
    max_depth=4,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=random_state
)

model_type3.fit(X_train_type3, y_train)

y_pred_enc_type3    = model_type3.predict(X_test_type3)
y_pred_type3 = le.inverse_transform(y_pred_enc_type3)


cm_type3        = confusion_matrix(y_test, y_pred_type3)
macro_f1        = f1_score(y_test, y_pred_type3, average="macro")
print("Macro F1 Score:", macro_f1)

# predict_proba: 각 샘플이 각 클래스일 확률
proba = model_type3.predict_proba(X_test_type3)  # shape: (n_samples, n_classes)

class_labels = model_type3.classes_  # 모델이 학습한 클래스 레이블
proba_df_type3 = pd.DataFrame(proba, columns=[f"prob_{c}" for c in class_labels], index=X_test.index)
proba_df_type3["true_label"] = y_test



### 2차

- Scaling_only + scaled_PCA

In [ ]:
X_train_type4 = pd.concat([X_train_type1, X_train_type2], axis=1)
X_test_type4 = pd.concat([X_test_type1, X_test_type2], axis=1)


model_type4 = XGBClassifier(
    objective="multi:softprob",
    num_class=6,  # 클래스 개수
    max_depth=4,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=random_state
)

model_type4.fit(X_train_type4, y_train)

y_pred_enc_type4    = model_type4.predict(X_test_type4)
y_pred_type4 = le.inverse_transform(y_pred_enc_type4)

cm_type4        = confusion_matrix(y_test, y_pred_type4)
macro_f1        = f1_score(y_test, y_pred_type4, average="macro")
print("Macro F1 Score:", macro_f1)

# predict_proba: 각 샘플이 각 클래스일 확률
proba = model_type4.predict_proba(X_test_type4)  # shape: (n_samples, n_classes)

class_labels = model_type4.classes_  # 모델이 학습한 클래스 레이블
proba_df_type4 = pd.DataFrame(proba, columns=[f"prob_{c}" for c in class_labels], index=X_test.index)
proba_df_type4["true_label"] = y_test

- Scaling_only + scaled_LDA

In [ ]:
X_train_type5 = pd.concat([X_train_type1, X_train_type3], axis=1)
X_test_type5 = pd.concat([X_test_type1, X_test_type3], axis=1)


model_type5 = XGBClassifier(
    objective="multi:softprob",
    num_class=6,  # 클래스 개수
    max_depth=4,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=random_state
)

model_type5.fit(X_train_type5, y_train)

y_pred_enc_type5    = model_type5.predict(X_test_type5)
y_pred_type5 = le.inverse_transform(y_pred_enc_type5)

cm_type5        = confusion_matrix(y_test, y_pred_type5)
macro_f1        = f1_score(y_test, y_pred_type5, average="macro")
print("Macro F1 Score:", macro_f1)

# predict_proba: 각 샘플이 각 클래스일 확률
proba = model_type5.predict_proba(X_test_type5)  # shape: (n_samples, n_classes)

class_labels = model_type5.classes_  # 모델이 학습한 클래스 레이블
proba_df_type5 = pd.DataFrame(proba, columns=[f"prob_{c}" for c in class_labels], index=X_test.index)
proba_df_type5["true_label"] = y_test

- scaled PCA + scaled LDA

In [ ]:
X_train_type6 = pd.concat([X_train_type2, X_train_type3], axis=1)
X_test_type6 = pd.concat([X_test_type2, X_test_type3], axis=1)


model_type6 = XGBClassifier(
    objective="multi:softprob",
    num_class=6,  # 클래스 개수
    max_depth=4,
    learning_rate=0.03,
    n_estimators=4000,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=random_state
)

model_type6.fit(X_train_type6, y_train)

y_pred_enc_type6    = model_type6.predict(X_test_type6)
y_pred_type6 = le.inverse_transform(y_pred_enc_type6)


cm_type6        = confusion_matrix(y_test, y_pred_type6)
macro_f1        = f1_score(y_test, y_pred_type6, average="macro")
print("Macro F1 Score:", macro_f1)

# predict_proba: 각 샘플이 각 클래스일 확률
proba = model_type6.predict_proba(X_test_type6)  # shape: (n_samples, n_classes)

class_labels = model_type6.classes_  # 모델이 학습한 클래스 레이블
proba_df_type6 = pd.DataFrame(proba, columns=[f"prob_{c}" for c in class_labels], index=X_test_type6.index)
proba_df_type6["true_label"] = y_test

### 3차

In [ ]:
X_train_type7 = pd.concat([X_train_type1, X_train_type6], axis=1)
X_test_type7 = pd.concat([X_test_type1, X_test_type6], axis=1)


model_type7 = XGBClassifier(
    objective="multi:softprob",
    num_class=6,  # 클래스 개수
    max_depth=4,
    learning_rate=0.03,
    n_estimators=4000,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=random_state
)

model_type7.fit(X_train_type7, y_train)

y_pred_enc_type7    = model_type7.predict(X_test_type7)
y_pred_type7 = le.inverse_transform(y_pred_enc_type7)


cm_type7        = confusion_matrix(y_test, y_pred_type7)
macro_f1        = f1_score(y_test, y_pred_type7, average="macro")
print("Macro F1 Score:", macro_f1)

# predict_proba: 각 샘플이 각 클래스일 확률
proba = model_type7.predict_proba(X_test_type7)  # shape: (n_samples, n_classes)

class_labels = model_type7.classes_  # 모델이 학습한 클래스 레이블
proba_df_type7 = pd.DataFrame(proba, columns=[f"prob_{c}" for c in class_labels], index=X_test_type7.index)
proba_df_type7["true_label"] = y_test

- Baseline: Scaled_PCA_LDA

In [ ]:
# ============== LDA           ==============
# ============== 최대5차원      ==============
n_lda_comps = 3
lda = LinearDiscriminantAnalysis(n_components=n_lda_comps)

X_train_type8 = pd.DataFrame(lda.fit_transform(X_train_type2, y_train),
                           columns=[f"LDA{i+1}" for i in range(n_lda_comps)],
                           index=X_train.index)
X_test_type8  = pd.DataFrame(lda.transform(X_test_type2),
                           columns=[f"LDA{i+1}" for i in range(n_lda_comps)],
                           index=X_test.index)


model_type8 = XGBClassifier(
    objective="multi:softprob",
    num_class=6,  # 클래스 개수
    max_depth=4,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=random_state
)



y_pred_enc_type8    = model_type8.predict(X_test_type8)
y_pred_type8 = le.inverse_transform(y_pred_enc_type8)

cm_type8        = confusion_matrix(y_test, y_pred_type8)
macro_f1        = f1_score(y_test, y_pred_type8, average="macro")
print("Macro F1 Score:", macro_f1)

# predict_proba: 각 샘플이 각 클래스일 확률
proba = model_type8.predict_proba(X_test_type8)  # shape: (n_samples, n_classes)

class_labels = model_type8.classes_  # 모델이 학습한 클래스 레이블
proba_df_type8 = pd.DataFrame(proba, columns=[f"prob_{c}" for c in class_labels], index=X_test.index)
proba_df_type8["true_label"] = y_test


In [ ]:
cm_list = [cm_type1,cm_type2,cm_type3,cm_type4,cm_type5,cm_type6,cm_type7]

for i in cm_list:
    plt.figure(figsize=(5, 4))
    sns.heatmap(i , annot=True, fmt="d", cmap="Blues",
            xticklabels=sorted(y.unique()), 
            yticklabels=sorted(y.unique()))
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()

In [ ]:
# predict_proba: 각 샘플이 각 클래스일 확률
proba = model_type7.predict_proba(X_test_type7)  # shape: (n_samples, n_classes)

# 결과를 DataFrame으로 변환
# 클래스 레이블이 0,1,2 라면, 컬럼명을 ["prob_0", "prob_1", "prob_2"] 등으로 설정
class_labels = model_type7.classes_  # 모델이 학습한 클래스 레이블
proba_df = pd.DataFrame(proba, columns=[f"prob_{c}" for c in class_labels], index=X_test_type7.index)

# 만약 실제 정답 열도 같이 두고 싶다면
proba_df["true_label"] = y_test

y_pred_type7 = model_type7.predict(X_test_type7)

# 5. 정답여부 열 추가
proba_df["Correct"] = np.where(y_pred_type7 == y_test, True, False)

# CSV로 저장
output_path = "predict_proba_results_112211.csv"
proba_df.to_csv(output_path, index=True)  # index=True이면 샘플 인덱스도 같이 저장

print(f"predict_proba 결과를 {output_path} 에 저장했습니다.")

In [ ]:
from sklearn.neighbors import NearestNeighbors

# 예시 DataFrame: df_hardclasses (target 컬럼 제외, 수치형만 있다고 가정)
X = df_hardclasses.drop(columns=["target"], errors="ignore")

# 1. 스케일링 (KNN은 스케일에 민감함)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. KNN 모델 정의
knn = NearestNeighbors(n_neighbors=6, metric="euclidean")  # 1개는 자기 자신이므로 6개 찾으면 이웃 5개
knn.fit(X_scaled)

# 3. 특정 인덱스 선택 (예: 42번 행)
idx = 439
sample = X_scaled[idx].reshape(1, -1)

# 4. 이웃 찾기
distances, indices = knn.kneighbors(sample)

# 5. 결과 확인
print("기준 샘플 인덱스:", idx)
print("이웃 인덱스:", indices[0][1:])  # 자기 자신 제외
print("이웃 거리:", distances[0][1:])

# 이웃 행값 가져오기
neighbors_df = df_hardclasses.iloc[indices[0][1:]]
print(neighbors_df)
